In [ ]:
## 2 cell đầu để chuẩn bị, ktra môi trường, path
from pathlib import Path
import sys

print("Python:", sys.version)
print("Working directory:", Path.cwd())


Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Working directory: d:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\notebooks


In [12]:
# Nếu notebook được chạy từ thư mục notebooks/
PROJECT_ROOT = Path("..").resolve()

MODEL_PATH = PROJECT_ROOT / "models" / "drowning_detection_yolov8" / "weights" / "best.pt"
VIDEO_PATH = PROJECT_ROOT / "test_vd" / "vdgocrong.mp4"

print("Project root:", PROJECT_ROOT)
print("Model:", MODEL_PATH)
print("Video:", VIDEO_PATH)

print("\nModel exists:", MODEL_PATH.exists())
print("Video exists:", VIDEO_PATH.exists())

if not MODEL_PATH.exists():
    print("\n⚠️ Chưa tìm thấy best.pt. Hãy đặt model mới vào đúng thư mục trước khi chạy tiếp.")

if not VIDEO_PATH.exists():
    print("\n⚠️ Chưa tìm thấy video. Hãy kiểm tra tên file trong test_vd/.")


Project root: D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system
Model: D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\models\drowning_detection_yolov8\weights\best.pt
Video: D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\test_vd\vdgocrong.mp4

Model exists: True
Video exists: True


In [13]:
## import YOLO, kiểm tra model
from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

print("Model loaded successfully.")
print("Class mapping:")
print(model.names)


Model loaded successfully.
Class mapping:
{0: 'Drowning', 1: 'Person out of water', 2: 'Swimming'}


In [14]:
## Test YOLO detection trên video - chưa dùng Bytetrack
# mục đích: tách lỗi detection và tracker
# nếu ở đây chưa detect được người/class hợp lý thì chưa nên đánh giá ByteTrack
# p/s: detect nhóm chưa tốt hẳn
detection_results = model.predict(
    source=str(VIDEO_PATH),
    conf=0.30,
    save=True,
    project=str(PROJECT_ROOT / "runs"),
    name="detect_video",
    exist_ok=True
)

print("Detection finished.")



WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1220) D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\test_vd\vdgocrong.mp4: 384x640 3 Swimmings, 124.2ms
video 1/1 (frame 2/1220) D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\test_vd\vdgocrong.mp4: 384x640 3 Swimmings, 62.8ms
video 1/1 (frame 3/1220) D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\test_vd\vdgocrong.mp4: 384x640 3 Swimmings, 61.9ms
video 1/1 (fra

In [15]:
## YOLOv8 + ByteTrack
# Ultralytics hỗ trợ ByteTrack bằng cách truyền tracker="bytetrack.yaml" vào model.track()
tracking_results = model.track(
    source=str(VIDEO_PATH),
    tracker="bytetrack.yaml",
    conf=0.30,
    save=True,
    project=str(PROJECT_ROOT / "runs"),
    name="track_bytetrack",
    exist_ok=True
)

print("Tracking finished.")



WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/1220) D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\test_vd\vdgocrong.mp4: 384x640 3 Swimmings, 60.5ms
video 1/1 (frame 2/1220) D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\test_vd\vdgocrong.mp4: 384x640 2 Swimmings, 94.6ms
video 1/1 (frame 3/1220) D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\test_vd\vdgocrong.mp4: 384x640 3 Swimmings, 62.3ms
video 1/1 (fram

In [16]:
## ktra Track ID
# Kiểm tra một số frame đầu tiên
for frame_idx, result in enumerate(tracking_results[:20]):
    print(f"Frame {frame_idx}:")

    if result.boxes.id is None:
        print("  Không có Track ID")
        continue

    track_ids = result.boxes.id.int().cpu().tolist()
    classes = result.boxes.cls.int().cpu().tolist()
    confidences = result.boxes.conf.cpu().tolist()

    for track_id, cls_id, conf in zip(track_ids, classes, confidences):
        class_name = model.names[int(cls_id)]
        print(f"  ID={track_id}, class={class_name}, conf={conf:.3f}")


Frame 0:
  ID=1, class=Swimming, conf=0.489
  ID=2, class=Swimming, conf=0.329
  ID=3, class=Swimming, conf=0.314
Frame 1:
  ID=1, class=Swimming, conf=0.505
  ID=3, class=Swimming, conf=0.302
Frame 2:
  ID=1, class=Swimming, conf=0.487
  ID=3, class=Swimming, conf=0.302
  ID=4, class=Swimming, conf=0.472
Frame 3:
  ID=3, class=Swimming, conf=0.424
  ID=4, class=Swimming, conf=0.352
Frame 4:
  ID=3, class=Swimming, conf=0.553
  ID=4, class=Swimming, conf=0.547
  ID=1, class=Swimming, conf=0.490
Frame 5:
  ID=3, class=Swimming, conf=0.477
  ID=4, class=Swimming, conf=0.479
  ID=1, class=Swimming, conf=0.618
Frame 6:
  ID=3, class=Swimming, conf=0.607
  ID=4, class=Swimming, conf=0.504
  ID=1, class=Swimming, conf=0.679
Frame 7:
  ID=3, class=Swimming, conf=0.461
  ID=4, class=Swimming, conf=0.615
  ID=1, class=Swimming, conf=0.711
Frame 8:
  ID=3, class=Swimming, conf=0.423
  ID=4, class=Swimming, conf=0.556
  ID=1, class=Swimming, conf=0.756
  ID=6, class=Swimming, conf=0.573
Frame 9:


In [17]:
## ktra bbox + class + conf + ID
if len(tracking_results) > 0:
    result = tracking_results[0]

    if result.boxes.id is not None:
        for i in range(len(result.boxes)):
            track_id = int(result.boxes.id[i].item())
            cls_id = int(result.boxes.cls[i].item())
            conf = float(result.boxes.conf[i].item())
            x1, y1, x2, y2 = result.boxes.xyxy[i].tolist()

            print({
                "frame": 0,
                "track_id": track_id,
                "class_id": cls_id,
                "class_name": model.names[cls_id],
                "confidence": round(conf, 3),
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "x2": round(x2, 2),
                "y2": round(y2, 2),
            })
    else:
        print("Frame đầu tiên không có Track ID.")


{'frame': 0, 'track_id': 1, 'class_id': 2, 'class_name': 'Swimming', 'confidence': 0.489, 'x1': 752.83, 'y1': 378.41, 'x2': 866.8, 'y2': 502.04}
{'frame': 0, 'track_id': 2, 'class_id': 2, 'class_name': 'Swimming', 'confidence': 0.329, 'x1': 767.8, 'y1': 397.72, 'x2': 857.59, 'y2': 501.43}
{'frame': 0, 'track_id': 3, 'class_id': 2, 'class_name': 'Swimming', 'confidence': 0.314, 'x1': 547.68, 'y1': 263.61, 'x2': 665.46, 'y2': 336.12}


In [20]:
## ghi tracking output thành CSV
import pandas as pd

tracking_rows = []

for frame_idx, result in enumerate(tracking_results):
    if result.boxes.id is None:
        continue

    boxes = result.boxes

    for i in range(len(boxes)):
        track_id = int(boxes.id[i].item())
        cls_id = int(boxes.cls[i].item())
        conf = float(boxes.conf[i].item())
        x1, y1, x2, y2 = boxes.xyxy[i].tolist()

        tracking_rows.append({
            "frame": frame_idx,
            "track_id": track_id,
            "class_id": cls_id,
            "class_name": model.names[cls_id],
            "confidence": conf,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
        })

tracking_df = pd.DataFrame(tracking_rows)

OUTPUT_DIR = PROJECT_ROOT / "runs" / "tracking_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

csv_path = OUTPUT_DIR / "vdgocrong_bytetrack.csv"
tracking_df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
print("Rows:", len(tracking_df))
tracking_df.head()


Saved: D:\Desktop\heThongThongMinh\HT_CanhbaoDuoinuoc\drowning-detection-system\runs\tracking_data\vdgocrong_bytetrack.csv
Rows: 703


,frame,track_id,class_id,class_name,confidence,x1,y1,x2,y2
0,0,1,2,Swimming,0.489275,752.828735,378.410645,866.799438,502.036438
1,0,2,2,Swimming,0.328962,767.800049,397.724060,857.589722,501.429504
2,0,3,2,Swimming,0.314431,547.676025,263.609314,665.463623,336.117859
3,1,1,2,Swimming,0.505388,753.132690,374.263733,873.023865,504.551880
4,1,3,2,Swimming,0.302019,550.222717,263.724945,664.530823,334.035675


In [19]:
## thống kê nhanh tracking
if tracking_df.empty:
    print("Không có tracking data.")
else:
    print("Số Track ID:", tracking_df["track_id"].nunique())

    print("\nSố frame theo từng Track ID:")
    print(
        tracking_df.groupby("track_id")["frame"]
        .nunique()
        .sort_values(ascending=False)
    )

    print("\nClass theo Track ID:")
    print(
        tracking_df.groupby("track_id")["class_name"]
        .agg(lambda x: ", ".join(sorted(set(x))))
    )


Số Track ID: 72

Số frame theo từng Track ID:
track_id
133    73
169    61
4      38
1      36
50     36
       ..
114     1
101     1
192     1
218     1
225     1
Name: frame, Length: 72, dtype: int64

Class theo Track ID:
track_id
1      Drowning, Swimming
2                Swimming
3                Swimming
4      Drowning, Swimming
6      Drowning, Swimming
              ...        
223              Swimming
224    Drowning, Swimming
225              Swimming
228              Drowning
234              Swimming
Name: class_name, Length: 72, dtype: str
